# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
#!pip install langchain_community langchain_experimental langchain_openai

In [4]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [ ]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
#!pip install faiss-cpu
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---------- ----------------------------- 5.0/18.2 MB 30.1 MB/s eta 0:00:01
   -------------------------- ------------- 12.1/18.2 MB 32.8 MB/s eta 0:00:01
   ---------------------------------------  18.1/18.2 MB 32.6 MB/s eta 0:00:01
   ---------------------------------------- 18.2/18.2 MB 30.1 MB/s  0:00:00


In [6]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [7]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [ ]:
#!pip install google-search-results

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32084 sha256=f9c00791af31d4855e886c10b77b5fff758d7aad2674b804628517e48c53aaff
  Stored in directory: c:\users\nicol\appdata\local\pip\cache\wheels\0c\47\f5\89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


  DEPRECATION: Building 'google-search-results' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google-search-results'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [15]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "you are a meal planer, you help with creating meals when given food items or suggest food items for a given meal: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need come up with meals or ingredients",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: Food items or meals. Output: a shopping list to create a meal of your choice or the meal the user chose. Please be very clear what the meal is and what other ingredients might be needed!",
    ),
]


prefix = """You are an AI who performs one task(suggest one meal + their missing ingredients) based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [16]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [17]:
OBJECTIVE = "What can i cook with my left-over rice?"

In [18]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [19]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should use the TODO action to create a list of ingredients needed for a meal
Action: TODO
Action Input: leftover rice
Observation: , chicken, and vegetables

As a meal planer, I would suggest using the leftover rice, chicken, and vegetables to create a delicious and balanced meal. Here are a few ideas:

1. Chicken Fried Rice: Heat a tablespoon of oil in a pan and add the leftover rice, chicken, and vegetables. Stir-fry for a few minutes and add soy sauce, garlic powder, and black pepper to taste. Serve hot.

2. Chicken and Vegetable Stir-Fry: Cut the leftover chicken and vegetables into bite-sized pieces. Heat oil in a pan and add the chicken and vegetables. Stir-fry for a few minutes and add soy sauce, ginger, and honey to taste. Serve over the leftover rice.

3. Chicken and Vegetable Soup: In a pot, add the leftover chicken, vegetables, and rice. Pour

{'objective': 'What can i cook with my left-over rice?'}